# N013 · 循环不变量、归纳与终止性

**目标：** 为简单算法写出初始化、保持和终止证明。

**先修：** N011。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 不变量；数学归纳法；递减量；部分正确性与终止性。

**配套讲解来源：** [同名逐章意见](../../comment/013_invariants_induction_termination.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

代码跑对了，你往往说不清"为什么对"；代码一改就悄悄错，你也不知道错在哪一环。这一章要解决的问题是：**学会给简单算法写一套"为什么它一定对、一定停"的论证**。工具就是三件套：循环不变量（每轮循环都保持为真的性质）、归纳法（从第一轮推到任意轮）、终止性（证明循环不会永远转下去）。

本章用两个小算法当练习场：

**问题 A：有序数组的插入位置。** 给一个从小到大排好序的数组 nums 和目标值 target，问 target 应该插在下标几，才能插进去之后数组仍然有序。

- 输入：`nums = [1, 3, 5]`，`target = 4`
- 输出：`2`
- 为什么是 2？因为 4 比 1 大、比 3 大、比 5 小，插在下标 2（3 和 5 之间）正合适：`[1, 3, 4, 5]`。

**问题 B：前缀最大值。** 给一个数组，输出每个位置"从头到这里为止见过的最大值"。

- 输入：`nums = [3, 1, 4, 2, 5]`
- 输出：`[3, 3, 4, 4, 5]`
- 为什么是它？到下标 0 为止最大是 3；到下标 1 为止 1 比 3 小，最大仍是 3；到下标 2 出现 4，最大变 4；到下标 3 时 2 没超过 4；到下标 4 出现 5，最大变 5。

两个问题都不难，重点是**给它们写出"初始化—保持—终止"三段式证明**，而不是只靠感觉说"显然对"。

## 2. 基础知识：先读懂这些词

- **循环不变量（loop invariant）**：在循环的每一轮开始（和结束）时都保持为真的一个性质。它像一句"合同"：合同不被违反，程序就没走歪。“思路推导”部分 特别提示，写不变量要同时说两头——"已经处理过的部分为什么是对的"以及"还没处理的部分里还剩哪些候选"，只说一头都不完整。
- **数学归纳法（mathematical induction）**：证明"性质在第 0 轮成立，且每轮成立就能推出下一轮成立"，于是它对所有轮都成立。用在循环上就是：初始时不变量真，每次推进不破坏它，那么循环全程它都真。
- **初始化（initiation）**：循环还没开始时（处理的部分为空），不变量是否天然成立。万事开头这一步最容易被忽略。
- **保持（maintenance / preservation）**：假设某一轮开始时不变量成立，走完这一轮后它仍然成立。这一步对应归纳法的"归纳步骤"。
- **终止性（termination）**：循环会在有限步内停下来。常用证据是找一个"递减量"——一个每轮严格减小/增大、又有下界/上界的量（比如索引 i 每轮加 1 且不能超过 len(nums)），它不可能无限变下去，循环必然停。
- **递减量（decreasing measure）**：就是上面那个"每轮严格变化且有界"的量，它是终止性的直接凭证。
- **部分正确性与终止性（partial correctness vs termination）**：拆开看，"如果循环停了，答案一定对"叫部分正确性（不变量 + 停止条件联合给出）；"循环一定会停"叫终止性。两件事合起来才是完全正确——一个算法可能"答案对但永远跑不完"，也可能"跑得完但答案是错的"，两条都要单独验。

## 3. 思路推导：从小例子开始

### 问题 A：linear_insert_position

- **Step 1：定策略——从左到右走，跳过所有比 target 小的元素。** 数组有序，所以"该插的位置"就是第一个"不小于 target 的元素"所在处（若没有这样的元素，就插在最末尾）。
- **Step 2：写出不变量。** 在循环的任意时刻：**指针 i 之前的所有元素（nums[0..i−1]）都小于 target；i 及之后的部分是尚未检查的候选。** 这正是 “思路推导”部分 说的"已处理部分正确 + 未处理部分候选"两头俱全。
- **Step 3：验证初始化。** i=0 时，"i 之前的元素"是空集，空集里所有元素都小于 target——空话为真，不变量天然成立。
- **Step 4：验证保持。** 每轮循环只有在 `nums[i] < target` 时才执行 `i+=1`，也就是说只有当前元素确实小于 target 我们才放它进"已处理区"；推进后"i 之前都小于 target"依旧成立。
- **Step 5：验证终止。** 每轮 i 严格加 1，而 i 有上界 len(nums)，最多 n 轮循环必停。
- **Step 6：读出答案。** 停止时只有两种可能：要么 `i == len(nums)`（所有元素都小于 target，插最末尾），要么 `nums[i] >= target`（首次遇到不小于 target 的元素）。两种情况合起来恰好是"第一个可插入位置"，返回 i 就是答案。

**手算演示（`nums=[1,3,5]`，`target=4`）：**

1. i=0：nums[0]=1 < 4，i 变 1。（不变量：0 之前即空集，成立）
2. i=1：nums[1]=3 < 4，i 变 2。（不变量：1 之前只有 1，1<4，成立）
3. i=2：nums[2]=5，5 < 4 不成立，循环停。返回 2。（不变量：2 之前是 1、3 都小于 4；停止原因是首次遇到不小于 4 的元素）

### 问题 B：prefix_max

- **Step 1：定策略。** 一边扫一边维护"到目前为止的最大值"current，每个位置把 current 记进结果。
- **Step 2：不变量。** 处理完第 k 个元素后，current 等于 nums[0..k] 中的最大值。
- **Step 3：初始化。** current=None 表示"还没见过任何元素"，第一个元素直接把 current 设为它自己（单元素的最大值就是它），不变量从第一步就成立。
- **Step 4：保持。** 若 current 已是前 k 个的最大值，那么前 k+1 个的最大值就是 max(current, 第 k+1 个)——代码里那行三目表达式干的就是这件事。
- **Step 5：终止。** for 循环走固定 n 个元素，天然终止。

**手算演示（`a=[3,1,4,2,5]`，也就是 “运行示例”部分 表格的数据）：**

| 前缀 | 最大值 |
|------|--------|
| [3] | 3 |
| [3,1] | 3 |
| [3,1,4] | 4 |
| [3,1,4,2] | 4 |
| [3,1,4,2,5] | 5 |

## 4. 核心代码：linear_insert_position

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def linear_insert_position(nums, target):
    i = 0
    while i < len(nums) and nums[i] < target:
        i += 1
    return i
```

### 逐段读懂代码

“分段实现”部分 定义了两个函数。

### 函数 1：`linear_insert_position` —— 线性找插入位置

```python
def linear_insert_position(nums, target):
    i = 0
    while i < len(nums) and nums[i] < target:
        i += 1
    return i
```

- `i=0`：指针从最左端出发，对应不变量的初始化（已处理部分为空）。
- `while i<len(nums) and nums[i]<target:`：两个条件缺一不可——`i<len(nums)` 是边界守卫，防止数组走穿了还去取 nums[i] 造成 IndexError；`nums[i]<target` 是业务条件，当前元素还比目标小就继续跳。注意 Python 的 and 是短路求值：先查前者，越界时根本不会碰 nums[i]。
- `i+=1`：合法推进——只有确认当前元素小于 target 才把它划入"已处理区"。
- `return i`：循环停止时 i 就是第一个可插入位置（理由见第三节 Step 6）。

### 函数 2：`prefix_max` —— 前缀最大值

```python
def prefix_max(nums):
    result = []
    current = None
    for x in nums:
        current = x if current is None else max(current, x)
        result.append(current)
    return result
```

- `result=[]`：输出列表，每个位置记一次"到此处为止的最大值"。
- `current=None`：用 None 表示"还没见过任何元素"。**为什么不初始化成 0？** 因为数组可能有负数（测试里就有 −4、−2），如果初值是 0，全负数数组的最大值会被错误地抬成 0。None 才是诚实的"尚未开始"。
- `current=x if current is None else max(current,x)`：一行三目表达式。第一次见元素（current 是 None）就把 current 设为它；之后每轮取"旧最大和新元素里较大的那个"。注意判断必须用 `is None` 而不是"current 是否为 0 之类的值"，原因同上。
- `result.append(current)`：把当前最大值收进结果。
- `return result`：空输入时循环不执行，返回空列表，行为自然正确。

“运行示例”部分 的 `show_table` 用 `a[:i+1]` 把每个前缀切出来，和 `prefix_max(a)` 的结果配对成第三节那张表——你可以逐行核对"前缀"和"最大值"是否吻合。

## 5. 分段实现：按顺序运行

**本章接口：** `linear_insert_position(nums, target)`、`prefix_max(nums)`

### linear_insert_position

In [1]:
def linear_insert_position(nums, target):
    i = 0
    while i < len(nums) and nums[i] < target:
        i += 1
    return i

### prefix_max

In [2]:
def prefix_max(nums):
    result = []
    current = None
    for x in nums:
        current = x if current is None else max(current, x)
        result.append(current)
    return result

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [3]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[3,1,4,2,5]
show_table(['前缀','最大值'],[(a[:i+1],m) for i,m in enumerate(prefix_max(a))])

前缀,最大值
[3],3
"[3, 1]",3
"[3, 1, 4]",4
"[3, 1, 4, 2]",4
"[3, 1, 4, 2, 5]",5


## 7. 正确性、适用条件与复杂度

循环初始时已处理前缀为空，性质真；每次合法推进保留性质；有限索引严格增加保证终止。停止条件联合不变量给出第一个可插入位置。

**代价：** 两个算法 O(n) 时间；插入位置 O(1) 辅助空间，前缀最大值 O(n) 输出空间。

### 展开理解

**为什么 `linear_insert_position` 是对的？** 我们把三段论证用完整的话说一遍。开头：i=0 时指针前面没有任何元素，"前面的元素都小于 target"这句话对空集来说是自动成立的。中间：每一轮循环放行一个元素的条件，正是"这个元素小于 target"，所以每推进一步，"i 之前全都小于 target"这条性质都不会被破坏；换句话说，循环走多久这条性质就保持多久。结尾：i 每轮加 1 又不能超过数组长度，所以循环最多 n 轮必停。停下时只有两种情况——数组走完了（所有元素都小于 target，返回的 i 恰好是末尾位置），或者当前元素不小于 target（它是从头数第一个这样的元素，插在它前面正好保序）。两种情况合起来，返回值永远是对的。这就同时完成了"若停则对"（部分正确性）和"必停"（终止性）两件事。

**为什么 `prefix_max` 是对的？** 同样三段：开头，第一个元素处理完，current 就是它自己，单元素的最大值确实是自己；保持，若 current 已是前 k 个的最大，那么加入第 k+1 个后的最大值，只能要么还是 current、要么是新的这个数，取两者较大必定正确；终止，for 循环次数固定为 n。所以输出的每一项都是"从头到该位置的最大值"。

**复杂度是多少？** 两个算法都是 O(n) 时间：每个元素被看一次、做一次比较。n=10⁵ 时大约十万次比较，一瞬间完成。空间上，插入位置只用了 i 一个变量，辅助空间 O(1)；前缀最大值要输出 n 个结果，输出空间 O(n)（这属于题目要求的输出，不算可省的浪费）。

给自己一个找不变量的小窍门，免得对着循环发呆：

1. 问"循环走完一半时，哪些变量分别代表什么"——把这个回答写下来，往往就是不变量。
2. 检查它有没有同时说到"已处理部分"和"未处理部分"两头（“思路推导”部分 的要求）。
3. 拿一个小输入走到循环中间，人工核对这句话那一刻是否为真——假了就改写。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 的每条断言：

- `assert linear_insert_position([1,3,5],4)==2`：**正常值测试（插中间）**。4 落在 3 和 5 之间，返回 2，验证"跳过小的、停在第一个不小于目标处"的主逻辑。
- `assert linear_insert_position([1,3,5],0)==0`：**边界值测试（比所有元素小）**。0 比头一个元素还小，应插在最前面，返回 0，验证第一轮就停的情况。
- `assert linear_insert_position([1,3,5],9)==3`：**边界值测试（比所有元素大）**。9 比所有元素大，走穿整个数组停在下标 3（末尾之后），验证 `i<len(nums)` 守卫把循环停在了正确位置且没越界。
- `assert linear_insert_position([],3)==0`：**边界值测试（空数组）**。空数组任何位置都是合法插入位，返回 0，验证零元素时循环根本不进、直接返回。
- `a=[-4,0,-2,5]; assert prefix_max(a)==[max(a[:i+1]) for i in range(len(a))]`：**交叉验证测试（含负数）**。对每个位置，用"重新对前缀调用内置 max"的笨办法算期望值，和 prefix_max 的输出逐项比对。特意包含 −4、−2 这样的负数，就是为了抓"current 误初始化为 0"那一类 bug——如果有那个 bug，第一个前缀 [−4] 的最大值会被错报成 0。
- `assert prefix_max([])==[]`：**边界值测试（空输入）**。空数组应返回空列表，防空输入崩溃。

归档一下六条断言的类型分布，你会发现插入位置把"目标值与数组的关系"三种情况（小、中、大）测全了：

| 断言 | 类型 | 它防的是哪类 bug |
|------|------|------------------|
| `[1,3,5],4 → 2` | 正常值（插中间） | 主逻辑走错位置 |
| `[1,3,5],0 → 0` | 边界（比所有元素小） | 第一轮就停时返回错 |
| `[1,3,5],9 → 3` | 边界（比所有元素大） | 走穿数组时越界/停错 |
| `[],3 → 0` | 边界（空数组） | 空输入崩溃 |
| 前缀最大值 vs 笨办法 | 交叉验证（含负数） | 初值误设为 0 |
| `prefix_max([])==[]` | 边界（空输入） | 空输入崩溃 |

In [4]:
assert linear_insert_position([1, 3, 5], 4) == 2

assert linear_insert_position([1, 3, 5], 0) == 0

assert linear_insert_position([1, 3, 5], 9) == 3

assert linear_insert_position([], 3) == 0

a = [-4, 0, -2, 5]

assert prefix_max(a) == [max(a[:i + 1]) for i in range(len(a))]

assert prefix_max([]) == []

print('N013: 所有本章断言通过')

N013: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 为前缀最大值写归纳证明。

**练习 2：** 构造越界错误最小反例。

<details>
<summary>展开思路提示（不是完整答案）</summary>

“练习”部分 的两道练习：

- **练习 1（为前缀最大值写归纳证明）**：照第三节的模板自己写三段，但用自己的话。提示结构：先写"命题 P(k)：处理完前 k 个元素后，current 等于这 k 个数的最大值"；然后第一段验证 P(1)（k=1 时 current=x 自身）；第二段写"假设 P(k) 成立，推导 P(k+1)"（关键一句：前 k+1 个的最大值只能出自"前 k 个的最大值"和"第 k+1 个数"两者之一）；第三段说明循环恰好执行 n 次所以终止。手算示例建议用 [2,7,1] 走一遍，期望输出 [2,7,7]。
- **练习 2（构造越界错误最小反例）**：故意把守卫条件写坏——比如把 `i<len(nums) and nums[i]<target` 改成只留后半句（先取 nums[i] 再比较）。提示：最小反例就是让数组走穿，输入 `nums=[]` 或 `nums=[1]`, `target=5` 都能触发 IndexError。先手推哪一轮会越界（nums=[1]、target=5：i=0 时 nums[0]<target 成立，i 变 1；下一轮要取 nums[1] 而长度只有 1——炸），再用 traceback 从最后一行往上定位到出错的那一行代码。边界情况记录：空数组与单元素数组各炸在哪一轮。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [5]:
# 完整实现：本单元格不依赖其他单元格。
def linear_insert_position(nums, target):
    i = 0
    while i < len(nums) and nums[i] < target:
        i += 1
    return i

def prefix_max(nums):
    result = []
    current = None
    for x in nums:
        current = x if current is None else max(current, x)
        result.append(current)
    return result

# 示例与回归测试
assert linear_insert_position([1, 3, 5], 4) == 2

assert linear_insert_position([1, 3, 5], 0) == 0

assert linear_insert_position([1, 3, 5], 9) == 3

assert linear_insert_position([], 3) == 0

a = [-4, 0, -2, 5]

assert prefix_max(a) == [max(a[:i + 1]) for i in range(len(a))]

assert prefix_max([]) == []

print('N013: 所有本章断言通过')

N013: 所有本章断言通过


## 11. 代表题与迁移

“题目映射”部分 列出的题目：

- **35. Search Insert Position（搜索插入位置，preview）**：这题就是 `linear_insert_position` 的原题——本章给的是 O(n) 线性版并配上完整证明；题目标签 preview 意味着后续会教二分 O(log n) 版，届时你要把同一个不变量换成二分版本再证一遍，正好检验本章学到的论证功夫。

照 notebook 原文提醒：以上条目用于知识映射，本章实现遵循教学契约，刷题时以 LeetCode 官方题面与评测为准。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。